# Using the CompositeFactoryClass Module in baseobjects.composition

## Introduction

`CompositeFactoryClass` is a specialized composite object that allows subclasses to act as pre-configured "presets" for a "head class." When a preset subclass is instantiated, it intercepts the construction process and returns an instance of the head class, pre-loaded with specific components.

This pattern is particularly useful for creating specialized configurations of a complex composite object without having to manually pass numerous component types each time.

This tutorial covers:
- Setting up a factory hierarchy
- Defining component presets
- Understanding the reverse-dispatching mechanism
- Customizing head class construction

**Prerequisites:**
- Familiarity with `BaseComposite` and `BaseDispatchingComposite`
- Understanding of `NamespaceRegisteredClass` and class registration
- Installed package: `baseobjects`

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

Import `CompositeFactoryClass` from `baseobjects.composition`. We also use `BaseComponent` for our example components and `NamespaceClassRegistry` for managing the hierarchy.



In [ ]:
from typing import Any, ClassVar
from baseobjects.composition import CompositeFactoryClass, BaseComponent
from baseobjects.classregistration import NamespaceClassRegistry



## Core Functionality

To use `CompositeFactoryClass`, you typically define a "Head Class" that inherits from it and sets `class_registration = True`. Then, you define preset subclasses that provide specific component configurations.

### Defining the Factory



In [ ]:
class MockComponent(BaseComponent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.name = kwargs.get("name", "Default")

class DataFactory(CompositeFactoryClass):
    """The head class for our data processing factory."""
    class_registration = True

class CSVFactory(DataFactory):
    """A preset for CSV data processing."""
    default_component_types = {"processor": (MockComponent, {"name": "CSV Processor"})}

class JSONFactory(DataFactory):
    """A preset for JSON data processing."""
    default_component_types = {"processor": (MockComponent, {"name": "JSON Processor"})}



## Module Interaction

`CompositeFactoryClass` interacts with the `NamespaceRegisteredClass` system to identify which class in the inheritance chain is the "Head Class". It then uses the `BaseComposite` system to populate that head class with components defined in the preset subclasses.


## Advanced Features

### Overriding Components at Runtime

You can still override the preset components by passing `component_types` to the constructor.



In [ ]:
class CustomProcessor(MockComponent): pass

# Override the preset component at runtime
custom_instance = CSVFactory(component_types={"processor": (CustomProcessor, {"name": "Custom CSV"})})
print(f"Processor type: {type(custom_instance.components['processor'])}")
print(f"Processor name: {custom_instance.components['processor'].name}")



### Customizing Head Class Construction

You can override `build_head_class` to inject additional arguments or modify the head class attributes during construction.



In [ ]:
class ExtendedDataFactory(DataFactory):
    # Start a new hierarchy for the extended factory
    class_registry: ClassVar[NamespaceClassRegistry | None] = None
    class_registration: ClassVar[bool] = True

    def __init__(self, *args, description="No description", **kwargs):
        super().__init__(*args, **kwargs)
        self.description = description

class DescribedCSVFactory(ExtendedDataFactory):
    default_component_types = {"processor": (MockComponent, {"name": "CSV"})}

    @classmethod
    def build_head_class(cls, head_class, *args, **kwargs):
        # Inject a default description if not provided
        kwargs.setdefault("description", "Processed CSV Data")
        return super().build_head_class(head_class, *args, **kwargs)

described_instance = DescribedCSVFactory()
print(f"Description: {described_instance.description}")



## Examples

Here is a simple example of using `CompositeFactoryClass` to create a theme manager where different themes are presets for a main `Theme` object.


In [ ]:
class ColorComponent(BaseComponent):
    def __init__(self, *args, color="white", **kwargs):
        super().__init__(*args, **kwargs)
        self.color = color

class Theme(CompositeFactoryClass):
    class_registration = True

class DarkTheme(Theme):
    default_component_types = {"background": (ColorComponent, {"color": "black"})}

class LightTheme(Theme):
    default_component_types = {"background": (ColorComponent, {"color": "white"})}

theme = DarkTheme()
print(f"Theme background: {theme.components['background'].color}")


### Introspecting Init Parameters

Subclasses of `CompositeFactoryClass` automatically gather information about their `__init__` method, which can be useful for introspection or dynamic processing.


In [ ]:
class ParametricFactory(DataFactory):
    def __init__(self, arg1, kwarg1="default", **kwargs):
        super().__init__(**kwargs)
        self.arg1 = arg1
        self.kwarg1 = kwarg1

print(f"Init Parameters: {list(ParametricFactory.init_parameters.keys())}")
print(f"Init Args: {ParametricFactory.init_args}")
print(f"Init Defaults: {ParametricFactory.init_defaults}")


## API Highlights

- **`CompositeFactoryClass`**: The base class for factory hierarchies.
  - `class_registration`: Must be set to `True` in the head class to enable dispatching.
  - `build_head_class(head_class, *args, **kwargs)`: Override this to customize how the head class is instantiated or to dynamically determine components.
  - `init_parameters`: A dictionary of the `__init__` parameters for the class.
  - `init_args`: A tuple of the names of the `__init__` positional and keyword arguments.
  - `init_defaults`: A dictionary of the default values for the `__init__` parameters.
  - `default_component_types`: Class attribute to define static component presets.



## Troubleshooting / FAQs

- **Problem**: Instantiating a subclass returns the subclass itself, not the head class.
  - **Solution**: Ensure that the head class (or an intermediate base class) has `class_registration = True`. Without registration, the dispatching mechanism in `__new__` cannot identify the head class.

- **Problem**: `AttributeError: 'NoneType' object has no attribute 'head_class'`
  - **Solution**: This usually happens when `class_registry` is `None`. Ensure that the class is properly registered within a hierarchy that has a designated head class.



## Conclusion and Next Steps

`CompositeFactoryClass` provides a clean way to manage complex composite configurations through a class-based preset system.

- **Next**: Learn more about `BaseDispatchingComposite` for standard forward-dispatching.
- **Reference**: See `src/baseobjects/composition/compositefactoryclass.py` for implementation details.

